# 01 — Data Audit: Section 1, Structural Checks

This notebook works through the **Structural** checks from §4 of the blueprint (`notebooks/subsea_sentinel_blueprint.md`). Before we build any features or models, we need to prove the raw file is what we think it is: the right shape, no duplicate observations, the right year range, and — because this is a *panel* (repeated observations of the same cables over time) — that it behaves like a real-world panel where entities can enter and exit, with the exception of six \"gappy\" cables we need to find and set aside later.

**Scope of this notebook (for now):** only the structural checks below. Target integrity, derived-column consistency, variable-meaning checks, and generator-artifact checks come later, as separate sections.

What we'll check, in order:
1. Row count is 4,821; 500 unique `cable_id` values
2. `cable_id` × `year` combinations are unique (no duplicate observations)
3. Years span 2015–2026
4. The panel is unbalanced (some cables enter after 2015, some exit before 2026)
5. Find the 6 cables with year gaps in their history"


## Setup

We use `pandas` (a library for working with tables of data, similar to a spreadsheet but scriptable) to load the CSV into a `DataFrame` — pandas' name for a table. Each row of `undersea_cables_master.csv` is one cable observed in one year."

In [ ]:
import pandas as pd

df = pd.read_csv('../data/undersea_cables_master.csv')
df = df.sort_values(['cable_id', 'year']).reset_index(drop=True)

df.head()

## Check 1 — Row count and unique cable count

**What we're testing:** the file should have exactly 4,821 rows (one row per cable per year it was observed), and exactly 500 distinct cables should be responsible for those rows.

**Why it matters:** everything downstream — feature engineering, train/test splits, model evaluation — assumes we know how many cables and how many cable-year observations exist. If the row count is off, either the file got truncated/duplicated on load, or our understanding of \"one row = one cable-year\" is wrong. Catching that now, with a loud `assert`, is much cheaper than discovering it after building features on top of bad data."

In [ ]:
n_rows = len(df)
n_unique_cables = df['cable_id'].nunique()

print(f'Row count: {n_rows}')
print(f'Unique cable_id count: {n_unique_cables}')

assert n_rows == 4_821, f'Expected 4821 rows, got {n_rows}'
assert n_unique_cables == 500, f'Expected 500 unique cables, got {n_unique_cables}'
print('PASS: row count and unique cable count match expectations.')

## Check 2 — `cable_id` × `year` is unique

**What we're testing:** for every (cable, year) pair, there should be at most one row. In other words, we should never see `CAB0001` appear twice for `2018`.

**Why it matters:** this is the *primary key* of the panel — the combination of columns that uniquely identifies a row. Later on, we'll do things like `groupby('cable_id').shift()` to get \"last year's fault status\" for each cable. If a (cable, year) pair were duplicated, that operation would silently produce two rows of history for one real year, corrupting every history-based feature downstream without raising an error."

In [ ]:
n_duplicate_keys = df.duplicated(subset=['cable_id', 'year']).sum()

print(f'Number of duplicated (cable_id, year) rows: {n_duplicate_keys}')

assert n_duplicate_keys == 0, 'Found duplicate (cable_id, year) rows — panel key is not unique!'
print('PASS: every (cable_id, year) pair appears exactly once.')

## Check 3 — Years span 2015–2026

**What we're testing:** the `year` column's minimum is 2015 and its maximum is 2026, with no years outside that range.

**Why it matters:** the entire modeling plan (§7 of the blueprint) is built around specific year ranges — dev folds from 2019–2024, a frozen holdout at 2025, and forward scoring for 2026. If the file actually contained, say, an extra row from 2027 or was missing 2026 entirely, every one of those downstream decisions would need to change. We confirm the boundary now so we can treat it as fixed later."

In [ ]:
min_year = df['year'].min()
max_year = df['year'].max()
years_present = sorted(df['year'].unique().tolist())

print(f'Min year: {min_year}')
print(f'Max year: {max_year}')
print(f'All years present: {years_present}')

assert min_year == 2015, f'Expected min year 2015, got {min_year}'
assert max_year == 2026, f'Expected max year 2026, got {max_year}'
print('PASS: years span 2015-2026.')

## Check 4 — The panel is unbalanced

**What we're testing:** a \"balanced panel\" would mean every one of the 500 cables has exactly one row for every year from 2015 to 2026 (12 rows each, 500 × 12 = 6,000 rows total). We only have 4,821 rows, so it can't be balanced — but we want to confirm *why*: some cables' first observed year is later than 2015 (they entered service after the panel started), and some cables' last observed year is earlier than 2026 (they exited — e.g. decommissioned, or simply not observed further). The blueprint's expected numbers are **188 cables entering after 2015** and **18 cables exiting before 2026**.

**Why it matters:** an unbalanced panel is completely normal for real infrastructure (cables get built and retired over time), but it has a direct consequence for modeling: we can't assume every cable has 12 years of history. Features like \"3-year rolling fault count\" need to handle cables with only 2 years of data. Confirming *how* unbalanced the panel is (which cables, how many) tells us how much of that edge-case handling we'll need."

In [ ]:
balanced_row_count = n_unique_cables * len(years_present)
print(f'A balanced panel would have {n_unique_cables} cables x {len(years_present)} years = {balanced_row_count} rows')
print(f'Actual row count: {n_rows} -> panel is unbalanced by {balanced_row_count - n_rows} rows\
')

# For each cable, find the first (min) and last (max) year it appears in the data.
# groupby('cable_id') splits the table into one mini-table per cable; ['year'] picks
# out just the year column from each; .agg(['min', 'max']) computes both the minimum
# and maximum year for each cable in one pass, returning a table indexed by cable_id
# with 'min' and 'max' columns.
cable_span = df.groupby('cable_id')['year'].agg(['min', 'max'])

entered_after_2015 = (cable_span['min'] > 2015).sum()
exited_before_2026 = (cable_span['max'] < 2026).sum()

print(f'Cables entering after 2015: {entered_after_2015}')
print(f'Cables exiting before 2026: {exited_before_2026}')

assert entered_after_2015 == 188, f'Expected 188 late entrants, got {entered_after_2015}'
assert exited_before_2026 == 18, f'Expected 18 early exits, got {exited_before_2026}'
print('PASS: panel is unbalanced as expected (188 late entrants, 18 early exits).')

## Check 5 — Find the 6 cables with year gaps

**What we're testing:** entering-late and exiting-early (Check 4) both produce a cable with a *contiguous* run of years — e.g. a cable observed 2018–2023 has 6 rows, and `max_year - min_year + 1 = 2023 - 2018 + 1 = 6`, which matches its row count. A \"gap\" cable is different: it's missing one or more years in the *middle* of its span. For example, a cable observed in every year from 2015–2024 and then again in 2026 (skipping 2025) has 11 rows, but `max_year - min_year + 1 = 2026 - 2015 + 1 = 12`. The row count (11) no longer matches the span (12) — that mismatch is exactly what exposes a gap.

**Why it matters:** the blueprint flags this explicitly (§0, correction #5 and §6, Step 4): a gap silently breaks any \"rolling window\" or \"previous year\" feature we build later, because pandas has no idea a year is missing — `groupby('cable_id').shift(1)` will happily hand you 2024's data and call it \"last year's value\" even when the actual previous row is from 2023. Finding these 6 cables now means we can make every history feature \"year-aware\" later, rather than discovering silently wrong numbers after modeling."

In [ ]:
# .size() counts how many rows fall into each group -- here, how many rows each
# cable_id has. This is our actual row count per cable.
cable_row_counts = df.groupby('cable_id').size()

# Reuse cable_span (min/max year per cable) from Check 4 and attach the row count
# and the \"expected\" row count if there were no gaps (span from min to max year,
# inclusive, hence the +1: e.g. 2015-2017 is 3 years, and 2017-2015+1 = 3).
cable_span = cable_span.copy()
cable_span['row_count'] = cable_row_counts
cable_span['expected_span'] = cable_span['max'] - cable_span['min'] + 1
cable_span['has_gap'] = cable_span['expected_span'] != cable_span['row_count']

gap_cable_ids = cable_span[cable_span['has_gap']].index.tolist()

print(f'Cables with year gaps: {len(gap_cable_ids)}\
')

assert len(gap_cable_ids) == 6, f'Expected 6 gap cables, found {len(gap_cable_ids)}'

for cable_id in gap_cable_ids:
    # Pull every year this specific cable was observed in, sorted low to high,
    # so we can see exactly where the missing year falls.
    cable_years = sorted(df.loc[df['cable_id'] == cable_id, 'year'])
    print(f'{cable_id}: {cable_years}')

print('\
PASS: found exactly 6 cables whose year range does not match their row count.')

## Section 1 summary

All five structural checks pass: 4,821 rows across 500 uniquely-identified cables, no duplicate (cable_id, year) rows, years spanning 2015–2026, an unbalanced panel (188 late entrants, 18 early exits), and 6 cables with a mid-history gap. The gap cables are not fixed here — per the blueprint (§6, Step 5), they get dropped later once we build the target column, since a gap breaks the `fault_next_year` label for the row right before the missing year.

**Next up (not in this notebook yet):** target integrity checks — §4's second checklist."